# 🧹 Script 2 — Data Cleaning + EDA (v3 — Word2Vec Embedding Added)
## Analisis Sentimen Konten Dakwah Penceramah Kota Bandung
**Author:** Fahmi Faturahman — STMIK Bandung 2026

---
### 📋 Alur Script (Diperbarui):
```
CELL 1 → Install library
CELL 2 → Mount Google Drive & Setup
CELL 3 → Load & cek data mentah (komentar + video)
CELL 4 → Cleaning raw_data_video.csv
CELL 5 → Cleaning raw_data_komentar.csv
CELL 6 →  Word Embedding (Word2Vec)
CELL 7 → EDA — Visualisasi
CELL 8 → Simpan semua data bersih & laporan
```
###  Input & Output:
- **Input:**  `raw_data_komentar.csv` + `raw_data_video.csv`
- **Output:** `clean_data_komentar.csv` + `clean_data_video.csv` + `word2vec_dakwah.model` + `comment_vectors_w2v.npy` + 9 grafik EDA

>  **Jalankan Script 1 terlebih dahulu sebelum menjalankan script ini!**
>  **Cell 6** melatih model **Word2Vec** dari komentar bersih, lalu menyimpan model + vektor rata-rata
> per komentar ke Drive. File ini tinggal di-*load* langsung di **Script 3** sebagai fitur numerik
> untuk model sentimen (ML klasik / deep learning), tanpa perlu training ulang embedding.

---
## Cell 1 — Install Library


In [1]:
!pip install -q PySastrawi
!pip install -q wordcloud
!pip install -q gensim scikit-learn
!pip install -q matplotlib seaborn pandas openpyxl

print('✅ Semua library berhasil diinstall!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.4 MB/s eta 0:00:00
✅ Semua library berhasil diinstall!


---
## Cell 2 — Mount Google Drive & Setup


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from collections import Counter
warnings.filterwarnings('ignore')

# Path
BASE_DIR          = '/content/drive/MyDrive/skripsi_dakwah_bandung'
INPUT_CSV         = os.path.join(BASE_DIR, 'raw_data_komentar.csv')
INPUT_VIDEO_CSV   = os.path.join(BASE_DIR, 'raw_data_video.csv')
OUTPUT_CSV        = os.path.join(BASE_DIR, 'clean_data_komentar.csv')
OUTPUT_VIDEO_CSV  = os.path.join(BASE_DIR, 'clean_data_video.csv')
W2V_MODEL_PATH    = os.path.join(BASE_DIR, 'word2vec_dakwah.model')      # 🆕 model Word2Vec (gensim)
W2V_VECTORS_PATH  = os.path.join(BASE_DIR, 'comment_vectors_w2v.npy')   # 🆕 vektor rata-rata per komentar
EDA_DIR           = os.path.join(BASE_DIR, 'eda_report')
os.makedirs(EDA_DIR, exist_ok=True)

# Style grafik
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 150})

# Inisialisasi Sastrawi stopword remover
factory_sw       = StopWordRemoverFactory()
stopword_remover = factory_sw.create_stop_word_remover()

print('✅ Setup selesai!')
print(f'📂 Input komentar   : {INPUT_CSV}')
print(f'📂 Input video      : {INPUT_VIDEO_CSV}')
print(f'📂 Output komentar  : {OUTPUT_CSV}')
print(f'📂 Output video     : {OUTPUT_VIDEO_CSV}')
print(f'📂 Output Word2Vec  : {W2V_MODEL_PATH}')
print(f'📂 Output vektor    : {W2V_VECTORS_PATH}')

Mounted at /content/drive
✅ Setup selesai!
📂 Input komentar   : /content/drive/MyDrive/skripsi_dakwah_bandung/raw_data_komentar.csv
📂 Input video      : /content/drive/MyDrive/skripsi_dakwah_bandung/raw_data_video.csv
📂 Output komentar  : /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data_komentar.csv
📂 Output video     : /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data_video.csv
📂 Output Word2Vec  : /content/drive/MyDrive/skripsi_dakwah_bandung/word2vec_dakwah.model
📂 Output vektor    : /content/drive/MyDrive/skripsi_dakwah_bandung/comment_vectors_w2v.npy


---
## Cell 3 —  Load & Cek Data Mentah
> Cek dulu kondisi data sebelum dibersihkan.

In [3]:
# ── Load data komentar ───────────────────────────────────────
df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')

# ── Load data video ───────────────────────────────────────────
df_video = pd.read_csv(INPUT_VIDEO_CSV, encoding='utf-8-sig')

print('='*60)
print('📊 DATA KOMENTAR (raw_data_komentar.csv)')
print('='*60)
print(f'Total baris    : {len(df):,}')
print(f'Kolom          : {list(df.columns)}')
print(f'Nilai null     :')
null_k = df.isnull().sum()
for col, n in null_k[null_k > 0].items():
    print(f'  {col:<30}: {n:,}')
print()

print('='*60)
print('📊 DATA VIDEO (raw_data_video.csv)')
print('='*60)
print(f'Total baris    : {len(df_video):,}')
print(f'Kolom          : {list(df_video.columns)}')
print(f'Nilai null     :')
null_v = df_video.isnull().sum()
for col, n in null_v[null_v > 0].items():
    print(f'  {col:<30}: {n:,}')
print()

print('5 baris pertama data video:')
display(df_video[['channel_name','title','topik_utama',
                   'ringkasan_konten','kesimpulan']].head())

📊 DATA KOMENTAR (raw_data_komentar.csv)
Total baris    : 21,611
Kolom          : ['comment_id', 'comment_text', 'author_name', 'like_count', 'published_at', 'video_id', 'channel_name', 'topik_utama']
Nilai null     :

📊 DATA VIDEO (raw_data_video.csv)
Total baris    : 162
Kolom          : ['channel_id', 'channel_name', 'video_id', 'title', 'description', 'published_at', 'view_count', 'like_count', 'comment_count', 'ada_transkrip', 'sumber_ringkasan', 'ringkasan_konten', 'topik_utama', 'kesimpulan', 'sumber_transkrip']
Nilai null     :
  channel_id                    : 9
  description                   : 6

5 baris pertama data video:


,channel_name,title,topik_utama,ringkasan_konten,kesimpulan
0,Ustadz Adi Hidayat,Kunjungan para Ustadz Sahabat UAH - Ustadz Adi...,Pendidikan Islam,Ceramah ini menampilkan silaturahmi antara Ust...,Pesan utama ceramah menekankan bahwa persatuan...
1,Ustadz Adi Hidayat,UAH Dilantik Jadi Stafsus? Hoax Atau Nyata? - ...,"Klarifikasi, Peran Ustadz, Prinsip Dakwah",Ustadz Adi Hidayat mengklarifikasi bahwa isu p...,Ceramah ini mengajak pendengar untuk melakukan...
2,Ustadz Adi Hidayat,Perbaiki Shalatmu Maka Allah Akan Permudah Uru...,Fiqih,Ustadz Adi Hidayat menjelaskan posisi krusial ...,Shalat merupakan ibadah fundamental yang tidak...
3,Ustadz Muhammad Faizar Hidayatullah,PENAMPAKAN POCONG DI RUMAH PANJI PETUALANG,Akidah,Ceramah ini menjelaskan gangguan spiritual di ...,Ceramah ini mengajarkan pemahaman Islam tentan...
4,Ustadz Muhammad Faizar Hidayatullah,MANTAN AHLI SANTET - RIA PUSPITA | PERNAH MAKA...,Pengalaman Mantan Ahli Santet,"Ceramah ini menampilkan kesaksian Ria Puspita,...",Ceramah ini memperingatkan akan bahaya dan tip...


---
## Cell 4 —  Cleaning `raw_data_video.csv`



In [4]:
# ============================================================
# CLEANING raw_data_video.csv (VERSION 2.1 - FULL DETAIL)
# ============================================================

df_v      = df_video.copy()
nv_awal   = len(df_v)
detail_v  = {}

print('='*60)
print('🧹 CLEANING DATA VIDEO')
print('='*60)

# ── STEP 1: Hapus duplikat video_id ─────────────────────────
n_sblm = len(df_v)
df_v = df_v.drop_duplicates(subset=['video_id'])
detail_v['Duplikat video_id'] = n_sblm - len(df_v)
print(f'[1] Hapus duplikat video_id     : -{detail_v["Duplikat video_id"]:,}')

# ── STEP 2: Hapus baris dengan title kosong/null ─────────────
n_sblm = len(df_v)
df_v = df_v.dropna(subset=['title'])
df_v = df_v[df_v['title'].str.strip() != '']
detail_v['Title kosong/null'] = n_sblm - len(df_v)
print(f'[2] Hapus title kosong/null     : -{detail_v["Title kosong/null"]:,}')

# ── STEP 3: Tandai & hapus topik 'Error' (Safe Logic) ────────
# Kita pakai apply(str) untuk menghindari error pd.NA di Pandas baru
mask_error_topik = df_v['topik_utama'].apply(lambda x: str(x).strip().lower() == 'error')
n_error_topik    = mask_error_topik.sum()
detail_v['Topik Error (Gemini gagal)'] = int(n_error_topik)

if n_error_topik > 0:
    print(f'[3] Topik bernilai Error        : {n_error_topik} video')
    print(f'    → Ditandai sebagai "Tidak Teridentifikasi"')
    df_v.loc[mask_error_topik, 'topik_utama'] = 'Tidak Teridentifikasi'
else:
    print(f'[3] Topik Error                : tidak ada ✅')

# ── STEP 4: Tandai ringkasan Error (Fix Three-Valued Logic) ──
def cek_invalid_ringkasan(val):
    """Fungsi Python murni agar logic OR stabil di semua versi Pandas"""
    if pd.isna(val) or not isinstance(val, str):
        return True
    t = val.strip().lower()
    return t.startswith('error') or t.startswith('tidak tersedia') or len(t) < 20

mask_error_ring = df_v['ringkasan_konten'].apply(cek_invalid_ringkasan)
n_error_ring = mask_error_ring.sum()
detail_v['Ringkasan Error/Kosong'] = int(n_error_ring)

if n_error_ring > 0:
    print(f'[4] Ringkasan Error/terlalu pendek : {n_error_ring} video')
    print(f'    → Ditandai sebagai "Tidak tersedia"')
    df_v.loc[mask_error_ring, 'ringkasan_konten'] = 'Tidak tersedia'
    df_v.loc[mask_error_ring, 'kesimpulan']       = 'Tidak tersedia'
else:
    print(f'[4] Ringkasan Error            : tidak ada ✅')

# ── STEP 5: Bersihkan kolom description ─────────────────────
def bersihkan_deskripsi(text):
    if not isinstance(text, str) or not text.strip(): return ''
    baris_bersih = []
    for baris in text.split('\n'):
        baris = baris.strip()
        if not baris: continue
        import re
        # Hapus promosi/kontak/link (Sesuai tahap Preprocessing Skripsi )
        ada_url = bool(re.search(r'http|www\.|bit\.ly|youtu\.be', baris, re.I))
        ada_wa  = bool(re.search(r'wa\.me|whatsapp|\+62|081|082|083|085|087|088', baris, re.I))
        terlalu_pendek = len(baris) < 15
        if not (ada_url or ada_wa or terlalu_pendek):
            baris_bersih.append(baris)
    hasil = ' '.join(baris_bersih)
    return hasil[:500] if hasil else ''

df_v['description_clean'] = df_v['description'].apply(bersihkan_deskripsi)
n_desc_bersih = (df_v['description_clean'].str.strip() != '').sum()
print(f'[5] Bersihkan description       : ✅ (Selesai)')

# ── STEP 6: Normalisasi published_at ───────────────────
df_v['published_at'] = pd.to_datetime(df_v['published_at'], errors='coerce').dt.strftime('%Y-%m-%d')
print(f'[6] Normalisasi published_at    : ✅ (Selesai)')

# ── STEP 7: Isi nilai null akhir ──────────────────
df_v['ada_transkrip']    = df_v['ada_transkrip'].fillna(False)
df_v['topik_utama']      = df_v['topik_utama'].fillna('Tidak Teridentifikasi')
df_v['ringkasan_konten'] = df_v['ringkasan_konten'].fillna('Tidak tersedia')
print(f'[7] Isi nilai null akhir        : ✅ (Selesai)')

nv_akhir = len(df_v)

# ── RINGKASAN FINAL ──────────────────────────────────────────
print('\n' + '='*60)
print('✅ CLEANING DATA VIDEO SELESAI')
print('='*60)
print(f'  Video sebelum cleaning : {nv_awal:,}')
print(f'  Video sesudah cleaning : {nv_akhir:,}')
print(f'  Video dihapus          : {nv_awal - nv_akhir:,}')
print('\n  Detail Perubahan:')
for alasan, n in detail_v.items():
    icon = '🗑️ ' if 'Duplikat' in alasan or 'kosong' in alasan else '🔧'
    print(f'  {icon} {alasan:<35}: {n}')

# ── DISTRIBUSI TOPIK (VISUALISASI) ───────────────────────────
print('\n📊 Distribusi Topik Dakwah (Setelah Cleaning):')
topik_dist = df_v['topik_utama'].value_counts()
for topik, n in topik_dist.items():
    pct = n / len(df_v) * 100
    bar = '█' * int(pct / 3) # Skala bar grafik
    print(f'  {topik:<35} {n:>4}  ({pct:>5.1f}%) {bar}')

print('\n📋 Preview Data Video Bersih:')
display(df_v[['channel_name','title','topik_utama','ringkasan_konten', 'kesimpulan']].head(5))

🧹 CLEANING DATA VIDEO
[1] Hapus duplikat video_id     : -0
[2] Hapus title kosong/null     : -0
[3] Topik Error                : tidak ada ✅
[4] Ringkasan Error            : tidak ada ✅
[5] Bersihkan description       : ✅ (Selesai)
[6] Normalisasi published_at    : ✅ (Selesai)
[7] Isi nilai null akhir        : ✅ (Selesai)

✅ CLEANING DATA VIDEO SELESAI
  Video sebelum cleaning : 162
  Video sesudah cleaning : 162
  Video dihapus          : 0

  Detail Perubahan:
  🗑️  Duplikat video_id                  : 0
  🗑️  Title kosong/null                  : 0
  🔧 Topik Error (Gemini gagal)         : 0
  🔧 Ringkasan Error/Kosong             : 0

📊 Distribusi Topik Dakwah (Setelah Cleaning):
  Akhlak                                29  ( 17.9%) █████
  Motivasi Islam                        26  ( 16.0%) █████
  Fiqih                                 14  (  8.6%) ██
  Akidah                                11  (  6.8%) ██
  Sirah Nabawiyah                        8  (  4.9%) █
  Sosial Kemasyarakatan  

,channel_name,title,topik_utama,ringkasan_konten,kesimpulan
0,Ustadz Adi Hidayat,Kunjungan para Ustadz Sahabat UAH - Ustadz Adi...,Pendidikan Islam,Ceramah ini menampilkan silaturahmi antara Ust...,Pesan utama ceramah menekankan bahwa persatuan...
1,Ustadz Adi Hidayat,UAH Dilantik Jadi Stafsus? Hoax Atau Nyata? - ...,"Klarifikasi, Peran Ustadz, Prinsip Dakwah",Ustadz Adi Hidayat mengklarifikasi bahwa isu p...,Ceramah ini mengajak pendengar untuk melakukan...
2,Ustadz Adi Hidayat,Perbaiki Shalatmu Maka Allah Akan Permudah Uru...,Fiqih,Ustadz Adi Hidayat menjelaskan posisi krusial ...,Shalat merupakan ibadah fundamental yang tidak...
3,Ustadz Muhammad Faizar Hidayatullah,PENAMPAKAN POCONG DI RUMAH PANJI PETUALANG,Akidah,Ceramah ini menjelaskan gangguan spiritual di ...,Ceramah ini mengajarkan pemahaman Islam tentan...
4,Ustadz Muhammad Faizar Hidayatullah,MANTAN AHLI SANTET - RIA PUSPITA | PERNAH MAKA...,Pengalaman Mantan Ahli Santet,"Ceramah ini menampilkan kesaksian Ria Puspita,...",Ceramah ini memperingatkan akan bahaya dan tip...


---
## Cell 5 —  Cleaning `raw_data_komentar.csv`
> Semua tahap preprocessing teks dijalankan di sini secara berurutan:
> **Hapus null → Hapus duplikat → Cleaning → Hapus stopword → Filter pendek**

In [5]:
# ── Fungsi-fungsi cleaning ──────────────────────────────────
def hapus_url(t):             return re.sub(r'http\S+|www\.\S+', ' ', t)
def hapus_mention(t):         return re.sub(r'[@#]\w+', ' ', t)
def hapus_emoji(t):
    p = re.compile('[' + u'\U0001F600-\U0001F64F'
        + u'\U0001F300-\U0001F5FF' + u'\U0001F680-\U0001F6FF'
        + u'\U0001F1E0-\U0001F1FF' + u'\U00002702-\U000027B0'
        + u'\U000024C2-\U0001F251' + ']+', flags=re.UNICODE)
    return p.sub(' ', t)
def hapus_nonalpha(t):         return re.sub(r'[^a-zA-Z\s]', ' ', t)
def case_folding(t):           return t.lower()
def norm_spasi(t):             return re.sub(r'\s+', ' ', t).strip()

def cleaning_pipeline(text):
    if not isinstance(text, str) or not text.strip(): return ''
    text = hapus_url(text)
    text = hapus_mention(text)
    text = hapus_emoji(text)
    text = hapus_nonalpha(text)
    text = case_folding(text)
    text = norm_spasi(text)
    return text

def hapus_stopword(text):
    return stopword_remover.remove(text) if text.strip() else ''

# ── Jalankan cleaning ────────────────────────────────────────
n_awal  = len(df)
detail  = {}

# Step 1: Hapus null & kosong
df = df.dropna(subset=['comment_text'])
df = df[df['comment_text'].str.strip() != '']
detail['Kosong / Null'] = n_awal - len(df)
print(f'[1] Hapus kosong/null       : -{detail["Kosong / Null"]:,} baris')

# Step 2: Hapus duplikat
n_sblm = len(df)
df = df.drop_duplicates(subset=['comment_text'])
detail['Duplikat'] = n_sblm - len(df)
print(f'[2] Hapus duplikat          : -{detail["Duplikat"]:,} baris')

# Step 3: Simpan panjang sebelum cleaning
df['panjang_raw']      = df['comment_text'].apply(lambda x: len(str(x)))
df['jumlah_kata_raw']  = df['comment_text'].apply(lambda x: len(str(x).split()))
print(f'[3] Hitung panjang raw      : ✅')

# Step 4: Cleaning pipeline
df['comment_clean'] = df['comment_text'].apply(cleaning_pipeline)
n_sblm = len(df)
df = df[df['comment_clean'].str.strip() != '']
detail['Kosong setelah cleaning'] = n_sblm - len(df)
print(f'[4] Cleaning pipeline       : -{detail["Kosong setelah cleaning"]:,} baris kosong setelah cleaning')

# Step 5: Hapus stopword
df['comment_no_stopword'] = df['comment_clean'].apply(hapus_stopword)
print(f'[5] Hapus stopword          : ✅')

# Step 6: Simpan panjang sesudah cleaning
df['panjang_clean']     = df['comment_clean'].apply(lambda x: len(str(x)))
df['jumlah_kata_clean'] = df['comment_clean'].apply(lambda x: len(str(x).split()))

# Step 7: Filter komentar terlalu pendek
n_sblm = len(df)
df = df[df['jumlah_kata_clean'] >= 3]
detail['Terlalu pendek'] = n_sblm - len(df)
print(f'[7] Filter <3 kata          : -{detail["Terlalu pendek"]:,} baris')

n_akhir = len(df)
print(f'\n✅ DATA CLEANING SELESAI!')
print(f'   Sebelum : {n_awal:,} komentar')
print(f'   Sesudah : {n_akhir:,} komentar')
print(f'   Dihapus : {n_awal - n_akhir:,} ({(n_awal-n_akhir)/n_awal*100:.1f}%)')

print('\nContoh hasil cleaning:')
display(df[['comment_text','comment_clean','comment_no_stopword']].head(3))

[1] Hapus kosong/null       : -0 baris
[2] Hapus duplikat          : -814 baris
[3] Hitung panjang raw      : ✅
[4] Cleaning pipeline       : -364 baris kosong setelah cleaning
[5] Hapus stopword          : ✅
[7] Filter <3 kata          : -1,528 baris

✅ DATA CLEANING SELESAI!
   Sebelum : 21,611 komentar
   Sesudah : 18,905 komentar
   Dihapus : 2,706 (12.5%)

Contoh hasil cleaning:


,comment_text,comment_clean,comment_no_stopword
0,"Andai saja hadir juga : Aa Gym, Buya Yahya, Ha...",andai saja hadir juga aa gym buya yahya habib ...,andai hadir aa gym buya yahya habib riziek ust...
1,Saya terlahir NU dan tumbuh dg segala aturan d...,saya terlahir nu dan tumbuh dg segala aturan d...,terlahir nu tumbuh dg aturan tata nu berkat us...
2,YANG NONTON INI SAMBIL NANGISS😭☝🏻☝🏻☝🏻 Allahumm...,yang nonton ini sambil nangiss allahumma sholl...,nonton nangiss allahumma sholli ala sayyidina ...


---
## Cell 6 —  Word Embedding dengan Word2Vec


In [6]:
# ============================================================
# WORD2VEC — WORD EMBEDDING UNTUK ANALISIS SENTIMEN (SCRIPT 3)
# ============================================================
from gensim.models import Word2Vec

# ── Step 1: Tokenisasi komentar bersih (pecah jadi list kata) ─
df['tokens'] = df['comment_no_stopword'].apply(lambda x: str(x).split())
n_token_kosong = int((df['tokens'].map(len) == 0).sum())
print(f'[1] Tokenisasi selesai        : {len(df):,} komentar')
print(f'    Token kosong (habis oleh stopword removal) : {n_token_kosong:,} baris')
print('    → Baris ini TETAP dipertahankan (tidak dihapus), nanti diberi vektor nol di Step 4')

# ── Step 2: Latih model Word2Vec (Skip-gram) ──────────────────
corpus_tokens = df['tokens'].tolist()

w2v_model = Word2Vec(
    sentences   = corpus_tokens,
    vector_size = 100,   # dimensi tiap vektor kata
    window      = 5,     # jumlah kata konteks kiri-kanan yang dilihat
    min_count   = 2,     # abaikan kata yang cuma muncul 1x (biar ga noise)
    sg          = 1,     # 1 = Skip-gram (lebih stabil utk korpus kecil-menengah), 0 = CBOW
    workers     = 4,
    epochs      = 20,
    seed        = 42
)

vocab_size = len(w2v_model.wv.key_to_index)
print(f'\n[2] Model Word2Vec dilatih')
print(f'    Ukuran vocabulary          : {vocab_size:,} kata unik')
print(f'    Dimensi vektor             : {w2v_model.vector_size}')
print(f'    Total kalimat (komentar)   : {len(corpus_tokens):,}')

# ── Step 3: Sanity check — apakah embedding "masuk akal"? ─────
# Cek kata-kata umum di konteks dakwah, lihat tetangga terdekatnya di ruang vektor
print('\n🔍 [3] Sanity check — kata paling mirip (uji makna semantik):')
kata_cek = ['allah', 'sholat', 'ustadz', 'bagus', 'jazakallah']
for kata in kata_cek:
    if kata in w2v_model.wv.key_to_index:
        mirip = w2v_model.wv.most_similar(kata, topn=5)
        mirip_str = ', '.join([f'{k} ({s:.2f})' for k, s in mirip])
        print(f'  "{kata:<12}" → {mirip_str}')
    else:
        print(f'  "{kata:<12}" → tidak ditemukan di vocabulary (mungkin < min_count)')

# ── Step 4: Ubah tiap komentar jadi 1 vektor (mean pooling) ───
# Kenapa mean pooling? Simple, cepat, dan jadi baseline yang solid untuk fitur ML klasik.
# Kalau nanti mau upgrade ke LSTM, tokens mentah + w2v_model.wv masih tersedia untuk dipakai.
def komentar_ke_vektor(tokens, model):
    vektor_kata = [model.wv[t] for t in tokens if t in model.wv.key_to_index]
    if len(vektor_kata) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vektor_kata, axis=0)

comment_vectors = np.array([
    komentar_ke_vektor(tok, w2v_model) for tok in df['tokens']
])
print(f'\n[4] Vektor rata-rata per komentar dibuat')
print(f'    Shape matriks vektor        : {comment_vectors.shape}')
print(f'    (baris = jumlah komentar, kolom = dimensi embedding = 100)')

# ── Step 5: Simpan model & vektor ke Google Drive ──────────────
w2v_model.save(W2V_MODEL_PATH)
np.save(W2V_VECTORS_PATH, comment_vectors)

print(f'\n✅ Model Word2Vec disimpan   : {W2V_MODEL_PATH}')
print(f'✅ Vektor komentar disimpan  : {W2V_VECTORS_PATH}')
print('   ⚠️ PENTING: urutan baris pada .npy 1:1 selaras dengan urutan baris df saat ini.')
print('   Kalau di Script 3 df diurutkan ulang/difilter lagi, load .npy ini SEBELUM')
print('   melakukan sorting/filtering tambahan pada df, atau simpan comment_id bareng vektornya.')

[1] Tokenisasi selesai        : 18,905 komentar
    Token kosong (habis oleh stopword removal) : 8 baris
    → Baris ini TETAP dipertahankan (tidak dihapus), nanti diberi vektor nol di Step 4

[2] Model Word2Vec dilatih
    Ukuran vocabulary          : 12,835 kata unik
    Dimensi vektor             : 100
    Total kalimat (komentar)   : 18,905

🔍 [3] Sanity check — kata paling mirip (uji makna semantik):
  "allah       " → hambanya (0.66), ampunan (0.66), kesendirian (0.66), mencurahkan (0.65), wataala (0.65)
  "sholat      " → tahajud (0.67), witir (0.65), dhuhur (0.65), subuh (0.65), ashar (0.64)
  "ustadz      " → ustad (0.68), ustdaz (0.60), riza (0.58), dimuliakan (0.57), ustadzuna (0.57)
  "bagus       " → terimksh (0.58), hihi (0.57), selali (0.56), conten (0.56), fresh (0.56)
  "jazakallah  " → jazakumulloh (0.85), khaiir (0.85), khair (0.85), katsira (0.83), khoir (0.83)

[4] Vektor rata-rata per komentar dibuat
    Shape matriks vektor        : (18905, 100)
    (baris = juml

---
## Cell 7 —  EDA — Visualisasi


In [7]:
def simpan(nama):
    path = os.path.join(EDA_DIR, nama)
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'  💾 Tersimpan: {nama}')

print('📊 Membuat grafik EDA...\n')

# ── GRAFIK 1: Sebelum vs Sesudah ────────────────────────────
print('Grafik 1: Sebelum vs Sesudah Cleaning')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('EDA 01 — Ringkasan Proses Data Cleaning', fontsize=13, fontweight='bold')
axes[0].bar(['Sebelum\nCleaning','Setelah\nCleaning'], [n_awal, n_akhir],
            color=['#C44E52','#55A868'], edgecolor='white', width=0.5)
for bar, val in zip(axes[0].patches, [n_awal, n_akhir]):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Jumlah Komentar', fontweight='bold')
axes[0].set_ylim(0, n_awal*1.2)
alasan = [k for k,v in detail.items() if v > 0]
jumlah = [v for v in detail.values() if v > 0]
if alasan:
    bars2 = axes[1].barh(alasan, jumlah, color='#4C72B0', edgecolor='white')
    axes[1].bar_label(bars2, fmt='{:,.0f}', padding=3, fontsize=10)
    axes[1].set_title('Data Dihapus per Alasan', fontweight='bold')
simpan('eda01_sebelum_sesudah.png')

# ── GRAFIK 2: Distribusi panjang komentar ───────────────────
print('Grafik 2: Distribusi Panjang Komentar')
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('EDA 02 — Distribusi Panjang & Jumlah Kata', fontsize=13, fontweight='bold')
for ax, col, color, label in [
    (axes[0,0], 'panjang_raw',     '#C44E52', 'Panjang Karakter (Raw)'),
    (axes[0,1], 'panjang_clean',   '#55A868', 'Panjang Karakter (Clean)'),
    (axes[1,0], 'jumlah_kata_raw', '#C44E52', 'Jumlah Kata (Raw)'),
    (axes[1,1], 'jumlah_kata_clean','#55A868','Jumlah Kata (Clean)'),
]:
    data = df[col].dropna()
    ax.hist(data, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),   color='navy',   linestyle='--', label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='-.', label=f'Median: {data.median():.1f}')
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=9)
simpan('eda02_distribusi_panjang.png')

# ── GRAFIK 3: Top penceramah ─────────────────────────────────
if 'channel_name' in df.columns:
    print('Grafik 3: Top 20 Penceramah')
    top = df['channel_name'].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(top.index[::-1], top.values[::-1], color='#4C72B0', edgecolor='white')
    ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=9)
    ax.set_title('EDA 03 — Top 20 Penceramah Berdasarkan Jumlah Komentar',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Jumlah Komentar')
    simpan('eda03_top_penceramah.png')

# ── GRAFIK 4: Distribusi topik ───────────────────────────────
if 'topik_utama' in df.columns:
    print('Grafik 4: Distribusi Topik Dakwah')
    topik = df['topik_utama'].value_counts()
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('EDA 04 — Distribusi Topik Dakwah', fontsize=13, fontweight='bold')
    colors = sns.color_palette('pastel', len(topik))
    axes[0].pie(topik.values, labels=topik.index, autopct='%1.1f%%',
               startangle=140, colors=colors)
    axes[0].set_title('Proporsi Topik (%)')
    bars = axes[1].bar(topik.index, topik.values, color=colors, edgecolor='white')
    axes[1].bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=9)
    axes[1].set_title('Jumlah Komentar per Topik')
    axes[1].tick_params(axis='x', rotation=30)
    simpan('eda04_distribusi_topik.png')

# ── GRAFIK 5: Tren bulanan ───────────────────────────────────
if 'published_at' in df.columns:
    print('Grafik 5: Tren Komentar per Bulan')
    df_t = df.copy()
    df_t['published_at'] = pd.to_datetime(df_t['published_at'], errors='coerce')
    df_t = df_t.dropna(subset=['published_at'])
    tren = df_t.groupby(df_t['published_at'].dt.to_period('M')).size().reset_index(name='n')
    tren['bulan_str'] = tren['published_at'].astype(str)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(range(len(tren)), tren['n'], marker='o', linewidth=2.5, color='#C44E52')
    ax.fill_between(range(len(tren)), tren['n'], alpha=0.12, color='#C44E52')
    ax.set_xticks(range(len(tren)))
    ax.set_xticklabels(tren['bulan_str'], rotation=45, ha='right', fontsize=9)
    ax.set_title('EDA 05 — Tren Jumlah Komentar per Bulan', fontsize=12, fontweight='bold')
    ax.set_xlabel('Bulan'); ax.set_ylabel('Jumlah Komentar')
    simpan('eda05_tren_bulanan.png')

# ── GRAFIK 6: Word Cloud ─────────────────────────────────────
print('Grafik 6: Word Cloud')
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('EDA 06 — Word Cloud Komentar', fontsize=13, fontweight='bold')
for series, ax, cmap, label in [
    (df['comment_clean'],       axes[0], 'Blues',  'Dengan Stopword'),
    (df['comment_no_stopword'], axes[1], 'Greens', 'Tanpa Stopword'),
]:
    teks = ' '.join(series.dropna().tolist())
    wc = WordCloud(width=900, height=500, background_color='white',
                   colormap=cmap, max_words=150, collocations=False).generate(teks)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(label, fontweight='bold')
simpan('eda06_wordcloud.png')

# ── GRAFIK 7: Top 30 kata ────────────────────────────────────
print('Grafik 7: Top 30 Kata Terbanyak')
semua_kata = ' '.join(df['comment_no_stopword'].dropna().tolist()).split()
semua_kata = [w for w in semua_kata if len(w) > 2]
top30 = Counter(semua_kata).most_common(30)
if top30:
    kata, freq = zip(*top30)
    fig, ax = plt.subplots(figsize=(12, 9))
    bars = ax.barh(list(kata)[::-1], list(freq)[::-1], color='#55A868', edgecolor='white')
    ax.bar_label(bars, fmt='{:,}', padding=3, fontsize=8)
    ax.set_title('EDA 07 — Top 30 Kata Terbanyak (Setelah Stopword Removal)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Frekuensi')
    simpan('eda07_top30_kata.png')

# ── GRAFIK 8: Heatmap penceramah vs topik ───────────────────
if 'channel_name' in df.columns and 'topik_utama' in df.columns:
    print('Grafik 8: Heatmap Penceramah vs Topik')
    top15 = df['channel_name'].value_counts().head(15).index
    pivot = df[df['channel_name'].isin(top15)].pivot_table(
        index='channel_name', columns='topik_utama', aggfunc='size', fill_value=0)
    fig, ax = plt.subplots(figsize=(14, 9))
    sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd',
               linewidths=0.5, ax=ax, cbar_kws={'label': 'Jumlah Komentar'})
    ax.set_title('EDA 08 — Heatmap: Penceramah vs Topik Dakwah',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    simpan('eda08_heatmap_penceramah_topik.png')

# ── GRAFIK 9: Proyeksi Word2Vec (PCA 2D) ─────────────────────
print('Grafik 9: Proyeksi Word2Vec (PCA 2D)')
from sklearn.decomposition import PCA

# Ambil top-N kata paling sering muncul yang ADA di vocabulary Word2Vec
top_kata_w2v = [w for w, _ in Counter(semua_kata).most_common(80)
                if w in w2v_model.wv.key_to_index][:60]

if len(top_kata_w2v) >= 5:
    vektor_top = np.array([w2v_model.wv[w] for w in top_kata_w2v])
    koord_2d   = PCA(n_components=2, random_state=42).fit_transform(vektor_top)

    fig, ax = plt.subplots(figsize=(13, 10))
    ax.scatter(koord_2d[:, 0], koord_2d[:, 1], s=60, color='#4C72B0', alpha=0.7, edgecolor='white')
    for (x, y), kata in zip(koord_2d, top_kata_w2v):
        ax.annotate(kata, (x, y), fontsize=9, xytext=(4, 4), textcoords='offset points')
    ax.set_title('EDA 09 — Proyeksi Word2Vec (PCA 2D) — Top 60 Kata Terbanyak',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Komponen PCA 1'); ax.set_ylabel('Komponen PCA 2')
    simpan('eda09_word2vec_pca.png')
else:
    print('  ⚠️ Kata dalam vocabulary Word2Vec terlalu sedikit, Grafik 9 dilewati.')

print('\n✅ Semua grafik EDA berhasil dibuat!')
print(f'📁 Tersimpan di: {EDA_DIR}/')

📊 Membuat grafik EDA...

Grafik 1: Sebelum vs Sesudah Cleaning
  💾 Tersimpan: eda01_sebelum_sesudah.png
Grafik 2: Distribusi Panjang Komentar
  💾 Tersimpan: eda02_distribusi_panjang.png
Grafik 3: Top 20 Penceramah
  💾 Tersimpan: eda03_top_penceramah.png
Grafik 4: Distribusi Topik Dakwah
  💾 Tersimpan: eda04_distribusi_topik.png
Grafik 5: Tren Komentar per Bulan
  💾 Tersimpan: eda05_tren_bulanan.png
Grafik 6: Word Cloud
  💾 Tersimpan: eda06_wordcloud.png
Grafik 7: Top 30 Kata Terbanyak
  💾 Tersimpan: eda07_top30_kata.png
Grafik 8: Heatmap Penceramah vs Topik
  💾 Tersimpan: eda08_heatmap_penceramah_topik.png
Grafik 9: Proyeksi Word2Vec (PCA 2D)
  💾 Tersimpan: eda09_word2vec_pca.png

✅ Semua grafik EDA berhasil dibuat!
📁 Tersimpan di: /content/drive/MyDrive/skripsi_dakwah_bandung/eda_report/


---
## Cell 8 — 💾 Simpan Semua Data Bersih & Laporan
> Simpan **komentar bersih** (+ vektor Word2Vec) dan **video bersih** ke Google Drive,
> lalu tampilkan laporan akhir.

In [8]:
# ── Simpan CSV komentar bersih ───────────────────────────────
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'✅ Komentar bersih disimpan : {OUTPUT_CSV}')

# ── Simpan CSV video bersih ──────────────────────────────────
df_v.to_csv(OUTPUT_VIDEO_CSV, index=False, encoding='utf-8-sig')
print(f'✅ Video bersih disimpan    : {OUTPUT_VIDEO_CSV}')

# ── Laporan teks ─────────────────────────────────────────────
lines = [
    '='*60,
    'LAPORAN CLEANING + EDA',
    'Skripsi Fahmi Faturahman — STMIK Bandung 2026',
    '='*60,
    '',
    '[ DATA KOMENTAR ]',
    f'  Sebelum cleaning : {n_awal:>8,}',
    f'  Sesudah cleaning : {n_akhir:>8,}',
    f'  Dihapus          : {n_awal - n_akhir:>8,}',
    f'  Data tersisa     : {n_akhir/n_awal*100:>7.1f}%',
    '',
    '[ DATA VIDEO ]',
    f'  Sebelum cleaning : {nv_awal:>8,}',
    f'  Sesudah cleaning : {nv_akhir:>8,}',
    f'  Dihapus          : {nv_awal - nv_akhir:>8,}',
    '',
    '[ DISTRIBUSI TOPIK DAKWAH (VIDEO BERSIH) ]',
]
for topik, n in df_v['topik_utama'].value_counts().items():
    lines.append(f'  {topik:<35} {n:>4}  ({n/len(df_v)*100:.1f}%)')

lines += [
    '',
    '[ WORD2VEC — WORD EMBEDDING ]',
    f'  Ukuran vocabulary   : {vocab_size:,} kata unik',
    f'  Dimensi vektor      : {w2v_model.vector_size}',
    f'  Shape vektor komentar: {comment_vectors.shape}',
    f'  Model disimpan      : {W2V_MODEL_PATH}',
    f'  Vektor disimpan     : {W2V_VECTORS_PATH}',
]

if 'channel_name' in df.columns:
    lines += ['', '[ TOP 10 PENCERAMAH — JUMLAH KOMENTAR ]']
    for r, (nama, jml) in enumerate(
        df['channel_name'].value_counts().head(10).items(), 1
    ):
        lines.append(f'  {r:>2}. {nama:<38} {jml:>6,}')

lines += [
    '',
    '[ OUTPUT FILE ]',
    f'  Komentar bersih : {OUTPUT_CSV}',
    f'  Video bersih    : {OUTPUT_VIDEO_CSV}',
    f'  Grafik EDA      : {EDA_DIR}/',
    '='*60,
]

report = '\n'.join(lines)
with open(os.path.join(EDA_DIR, 'eda_summary.txt'), 'w', encoding='utf-8') as f:
    f.write(report)

print('\n' + report)
print(f'\n✅ Laporan tersimpan!')
print('\n🎉 Script 2 selesai! Data bersih + Word2Vec embedding siap dipakai di Script 3.')

✅ Komentar bersih disimpan : /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data_komentar.csv
✅ Video bersih disimpan    : /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data_video.csv

LAPORAN CLEANING + EDA
Skripsi Fahmi Faturahman — STMIK Bandung 2026

[ DATA KOMENTAR ]
  Sebelum cleaning :   21,611
  Sesudah cleaning :   18,905
  Dihapus          :    2,706
  Data tersisa     :    87.5%

[ DATA VIDEO ]
  Sebelum cleaning :      162
  Sesudah cleaning :      162
  Dihapus          :        0

[ DISTRIBUSI TOPIK DAKWAH (VIDEO BERSIH) ]
  Akhlak                                29  (17.9%)
  Motivasi Islam                        26  (16.0%)
  Fiqih                                 14  (8.6%)
  Akidah                                11  (6.8%)
  Sirah Nabawiyah                        8  (4.9%)
  Sosial Kemasyarakatan                  6  (3.7%)
  Pendidikan Islam                       4  (2.5%)
  Tafsir Quran                           4  (2.5%)
  Kesehatan Islam                   